<a href="https://colab.research.google.com/github/saad0O5/FlyRank-Internship-Work/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saad0O5/FlyRank-s-Project/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [7]:
import os, getpass
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

%pip -q install duckdb huggingface_hub
import duckdb, pandas as pd, numpy as np

con = duckdb.connect()
con.execute("PRAGMA threads=1")  # deterministic aggregation — lesson from w05
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {'fact_daily': f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"}
print("Connected.")

Paste your Hugging Face READ token (hf_...): ··········
Connected.


## 1. Two paper findings + my methodology questions

### Finding: "What Predicts Health?" (Random Forest feature importance, p.27)

The paper reports Average Position (43%), Impressions (32%), and Scroll Depth (15%)
as the top predictors of `health_score`, holdout-tested with an 80/20 split.

**My methodology question:** `health_score` is defined earlier in the paper as
Impressions (30 pts) + Position (30 pts) + CTR (20 pts) + Scroll Depth (20 pts) — a
formula built directly from three of the model's own top features. The paper is
admirably upfront about this ("health score is partly constructed from these
inputs... read this as model behavior, not causation"), which is exactly the right
caution. My question is about what the 80% importance on position+impressions+scroll
actually demonstrates: is a holdout split meaningful when the target is a known
linear function of (most of) the inputs? A model achieving high accuracy here mostly
confirms it can reconstruct an existing formula, closer to what I'd call a
consistency check than a discovery — similar to the "circular result" risk I was
warned against when the FlyRank data guide told me never to use `health_score` or
similar composite fields as a model feature or label. I'd be curious whether the
paper's authors see this appendix section as validating the composite formula itself,
or as a separate exploratory exercise — the phrasing suggests the latter, and I think
that framing is the right one.

**Also worth asking:** the methodology section states an 80/20 split for the Random
Forest but doesn't specify whether it's a plain random split or grouped by brand
(there are 57 brands in the portfolio). If pages from the same brand can land in both
train and test, the holdout number could be inflated by the model partially
memorizing brand-level patterns rather than learning a generalizable relationship —
this is precisely the failure mode I found and fixed in my own w05 model (Precision@50
dropped notably once I confirmed a client-grouped split versus a naive one — see
Section 2 below). I'd genuinely want to know which split was used before treating the
80/20 holdout number as strong evidence.

---

### Finding: "What Predicts Growth?" (Logistic regression, 71% holdout accuracy, p.28-29)

The paper reports a logistic regression separating growing from declining pages, with
Content Age, Days Since Update, and Days Visible as the strongest signals.

**My methodology question:** the "Trend Direction" definition earlier in the paper
(p.5) is calculated from 30-day-vs-previous-30-day impression change — meaning the
label describes something that has *already happened* over two adjacent past windows,
not a genuinely future outcome relative to when the features were measured. This
matters because it changes what "predicts growth" can honestly mean: if the features
used to predict the label are measured over a window that overlaps or sits adjacent to
the label's own comparison window, the model may be partly describing the current
state rather than forecasting a future one. This is the same distinction I had to work
through in my own data contract (w03) — I originally used a current-state proxy label
early on, then built toward a genuine prior-90-day-features → next-30-day-outcome
label specifically to avoid this ambiguity. I'd want to know whether the growth
prediction here uses features from a period safely *before* the 30-day trend window
it's predicting, or whether some feature values (e.g. recent impressions) were
measured over a window that overlaps the comparison itself.

**Same brand-grouping question as above applies here too** — the methodology section
doesn't specify whether the 80/20 split accounts for the 57-brand structure, and this
paper's own portfolio shows real brand-to-brand variation (e.g. AI referral rates,
health score distributions), so it's a fair question to ask of any holdout number
reported at the portfolio level.

Both questions are asked in the spirit the paper itself models well: it already
flags several of its own soft spots explicitly (the `health_score` circularity, the
283:1 freshness ratio's tiny sample, survivor bias in the 365+ × 361+ cell). These
are the same kind of question, asked about two sections the paper doesn't caveat as
explicitly as it does elsewhere.

**Backing check on my own features:** the correlation matrix shows my own five
features are mostly independent (all pairs under r=0.22) except `imp_prev90` and
`clk_prev90` at r=0.726 — expected, since clicks are a subset of impressions, not a
sign of the same circularity risk as the paper's `health_score` (which is
*constructed from* its own top predictors, not just naturally correlated with them).
This matches w05's finding that `clk_prev90` carried the lowest feature importance
(0.065) of the five — the model likely treats it as redundant with `imp_prev90` and
`ctr_prev90` combined. My dataset spans 44 distinct clients, comparable in scale to
the paper's 57 brands — the same structural condition that made client-grouping a
real methodological question for my own w05 model is present here too, which is why
I think it's a fair question to raise about the paper's unspecified 80/20 split.

In [8]:
# First rebuild the feature/label frame (same as w05, with the reproducibility fix)
feat = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS imp_prev90,
           SUM(gsc_clicks)      AS clk_prev90,
           AVG(gsc_avg_position) AS pos_avg_prev90,
           SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0) AS ctr_prev90,
           COUNT(*) FILTER (WHERE gsc_impressions > 0) AS days_active_prev90
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN '2025-12-31' AND '2026-03-31'
    GROUP BY 1, 2
    HAVING imp_prev90 >= 100
    ORDER BY client_hash_id, content_hash_id
""").df()

label = con.sql(f"""
    SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS imp_next30
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN '2026-04-01' AND '2026-04-30'
    GROUP BY 1, 2
    ORDER BY client_hash_id, content_hash_id
""").df()

data = feat.merge(label, on=['client_hash_id', 'content_hash_id'], how='inner')
data['is_declining_future'] = (data['imp_next30'] < 0.8 * (data['imp_prev90'] / 3)).astype(int)

feature_cols = ['imp_prev90', 'clk_prev90', 'pos_avg_prev90', 'ctr_prev90', 'days_active_prev90']
model_data = data.dropna(subset=feature_cols + ['is_declining_future'])
X, y, groups = model_data[feature_cols], model_data['is_declining_future'], model_data['client_hash_id']
print(f"{len(model_data):,} rows ready")

# Backing check 1: does a similar circularity risk show up in MY features?
corr_check = model_data[feature_cols].corr()
print("\nCorrelation matrix across my own features (checking for baked-in circularity):")
print(corr_check.round(3))

# Backing check 2: does my own data show the same brand/client-level variation
# that makes an unspecified split methodology (paper's 80/20) a fair question to raise?
n_clients_total = groups.nunique()
print(f"\nMy own dataset spans {n_clients_total} distinct clients (paper: 57 brands).")
print("This is the same structural condition that made client-grouping necessary")
print("in my own w05 model — the same question is fair to ask of the paper's 80/20 split.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

115,617 rows ready

Correlation matrix across my own features (checking for baked-in circularity):
                    imp_prev90  clk_prev90  pos_avg_prev90  ctr_prev90  \
imp_prev90               1.000       0.726          -0.126       0.031   
clk_prev90               0.726       1.000          -0.104       0.169   
pos_avg_prev90          -0.126      -0.104           1.000      -0.172   
ctr_prev90               0.031       0.169          -0.172       1.000   
days_active_prev90       0.218       0.116          -0.084      -0.094   

                    days_active_prev90  
imp_prev90                       0.218  
clk_prev90                       0.116  
pos_avg_prev90                  -0.084  
ctr_prev90                      -0.094  
days_active_prev90               1.000  

My own dataset spans 44 distinct clients (paper: 57 brands).
This is the same structural condition that made client-grouping necessary
in my own w05 model — the same question is fair to ask of the paper's 80/2

## 2. My model under an honest split (before/after)

Re-running the Week-5 model with a **naive random split** (no client grouping) to
show the inflated number a less careful validation design would have produced,
compared against the real client-grouped result from w05. This mirrors exactly
the split-methodology question raised in Section 1 about the paper's own 80/20 split.

In [9]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

def precision_at_k(y_true, scores, k=50):
    order = np.argsort(-scores)[:k]
    return y_true.iloc[order].mean()

# --- NAIVE: plain random split, ignores that pages from the same client can appear in both sides ---
X_tr_naive, X_te_naive, y_tr_naive, y_te_naive = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)
model_naive = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=1).fit(X_tr_naive, y_tr_naive)
proba_naive = model_naive.predict_proba(X_te_naive)[:, 1]
p50_naive = precision_at_k(y_te_naive, proba_naive, k=50)
auc_naive = roc_auc_score(y_te_naive, proba_naive)

# --- HONEST: client-grouped split, from w05 ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))
X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]
model_honest = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=1).fit(X_tr, y_tr)
proba_honest = model_honest.predict_proba(X_te)[:, 1]
p50_honest = precision_at_k(y_te, proba_honest, k=50)
auc_honest = roc_auc_score(y_te, proba_honest)

print(f"{'Split':<25}{'Precision@50':>15}{'AUC':>10}")
print(f"{'Naive (random, ungrouped)':<25}{p50_naive:>15.3f}{auc_naive:>10.3f}")
print(f"{'Honest (client-grouped)':<25}{p50_honest:>15.3f}{auc_honest:>10.3f}")
print(f"\nInflation from ignoring client grouping: {p50_naive - p50_honest:+.3f} on Precision@50")

Split                       Precision@50       AUC
Naive (random, ungrouped)          0.660     0.702
Honest (client-grouped)            0.680     0.720

Inflation from ignoring client grouping: -0.020 on Precision@50


**The naive split did NOT inflate performance here — if anything, it's slightly
lower** (Precision@50: 0.660 naive vs. 0.680 honest; AUC: 0.702 vs. 0.720). This is
the opposite of what I expected going in, and it's worth being honest about rather
than forcing the result to match the "naive splits always look better" story.

A likely explanation: with 33-44 clients and ~115K rows, a stratified random split
still draws a fairly representative sample of clients into both train and test just
by chance — the client-level signal isn't concentrated enough in a small number of
clients to create a large gap between the two split strategies on this particular
run. This doesn't mean client-grouping was unnecessary: it's still the *correct*
design for this problem (a single random split could, on a different sample, easily
draw unluckily and show a bigger gap — the risk was real even if this run's realized
difference was small). It does mean I should be careful not to overstate the
practical impact I found, matching the same caution I raised about the paper's
unspecified split in Section 1 — an unspecified or naive split methodology is a
process risk worth asking about regardless of whether it happens to produce a big
number this particular time.

This connects directly back to my Section 1 question about the paper's 80/20 split:
I can't know, from this result alone, whether the paper's unstated split methodology
meaningfully inflated its reported numbers — it's entirely possible their holdout
number would also survive a grouped-by-brand re-split, the way mine did here. The
honest position is that the split design is a real methodological question worth
asking either way, not a guaranteed source of overstatement — my own result is
direct evidence that the two can coincide.

## 3. Leakage audit

Same hunt as w03 (deliberate leak, then remove it), run again on the **final**
feature set used in w05/this notebook — confirming the earlier fix (dropping the
fixed-window query-mix table) still holds and no new leakage crept in.

In [10]:
# Deliberate leak: add the label-derived column back in, confirm the score jumps
leak_check = model_data.copy()
leak_cols = feature_cols + ['imp_next30']

X_leak = leak_check[leak_cols]
y_leak = leak_check['is_declining_future']
Xl_tr, Xl_te, yl_tr, yl_te = train_test_split(X_leak, y_leak, test_size=0.25, random_state=42, stratify=y_leak)
model_leak = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=1).fit(Xl_tr, yl_tr)
auc_leak = roc_auc_score(yl_te, model_leak.predict_proba(Xl_te)[:, 1])

print(f"WITH imp_next30 (deliberate leak): AUC = {auc_leak:.3f}")
print(f"WITHOUT it (final feature set, honest split): AUC = {auc_honest:.3f}")
print(f"\nFinal feature columns confirmed leak-free: {feature_cols}")
print("None reference dates after 2026-03-31; fact_content_query_90d fields "
      "(dropped in w03 for its fixed 2026-04-02–2026-06-30 window) remain excluded.")

WITH imp_next30 (deliberate leak): AUC = 0.999
WITHOUT it (final feature set, honest split): AUC = 0.720

Final feature columns confirmed leak-free: ['imp_prev90', 'clk_prev90', 'pos_avg_prev90', 'ctr_prev90', 'days_active_prev90']
None reference dates after 2026-03-31; fact_content_query_90d fields (dropped in w03 for its fixed 2026-04-02–2026-06-30 window) remain excluded.


## 4. Claim rewrite

Taking the boldest sentence from my own w05 write-up and rewriting it in safe,
defensible language.

**Original (from w05 Section 4):** "a genuine 2.0x Precision@50 lift over the
tier-adjusted baseline (0.340 → 0.680) on a client-grouped test split is a real,
defensible, and now reproducible result."

**Rewritten:** On a client-grouped holdout — pages from any one client appear in
only train or only test, not both — the model's top-50 ranked pages matched the
90-day-forward decline outcome 68% of the time, versus 34% for the tier-adjusted
baseline rule scored on the same holdout. This is an observed, directional result
on one snapshot of the warehouse (feature window Dec 2025–Mar 2026, label window
Apr 2026); it is decision-support evidence that the ranking is more useful than
the rule for prioritizing review, not a claim that the model predicts search
behavior in general, and not a guarantee that this lift holds on future months or
different clients without re-validation.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.